In [1]:
import pandas as pd

In [2]:
Selected_features_Q2 = pd.read_csv(r'C:\Users\Ahmed\Documents\GazaSkyGeeksOlmpics\project\tests\Fares\data\processed\Selected_features_Q2.csv')

Selected_features_Q2.head()


,avg_score_until_cutoff,submitted_assessments_until_cutoff,arab_active_days_equivalent_until_cutoff,avg_submission_delay_arab_days_until_cutoff,homepage,forumng,unique_sites_until_cutoff,unique_activity_types_until_cutoff,clicks_per_active_day_until_cutoff,resource,subpage,url,oucontent,quiz,highest_education,target
0,81.000000,3.0,16.231343,-0.746269,106.0,146.0,43.0,6.0,24.482759,10.0,27.0,2.0,419.0,0.0,HE Qualification,successful
1,69.333333,3.0,27.985075,0.932836,234.0,333.0,63.0,7.0,19.720000,7.0,75.0,43.0,291.0,0.0,HE Qualification,successful
2,0.000000,0.0,6.716418,0.000000,59.0,126.0,22.0,6.0,23.416667,4.0,22.0,4.0,66.0,0.0,A Level or Equivalent,at_risk
3,72.333333,3.0,41.977612,-1.305970,297.0,358.0,61.0,7.0,18.426667,11.0,105.0,61.0,549.0,0.0,A Level or Equivalent,successful
4,54.000000,3.0,26.305970,6.529851,152.0,162.0,52.0,7.0,17.000000,32.0,42.0,8.0,401.0,0.0,Lower Than A Level,successful


## Step 1: Create a working copy
Use `Selected_features_Q1_copy` for all analysis and preprocessing so the original dataframe stays unchanged.


In [3]:
Selected_features_Q2_copy = Selected_features_Q2.copy()
Selected_features_Q2_copy.head()


,avg_score_until_cutoff,submitted_assessments_until_cutoff,arab_active_days_equivalent_until_cutoff,avg_submission_delay_arab_days_until_cutoff,homepage,forumng,unique_sites_until_cutoff,unique_activity_types_until_cutoff,clicks_per_active_day_until_cutoff,resource,subpage,url,oucontent,quiz,highest_education,target
0,81.000000,3.0,16.231343,-0.746269,106.0,146.0,43.0,6.0,24.482759,10.0,27.0,2.0,419.0,0.0,HE Qualification,successful
1,69.333333,3.0,27.985075,0.932836,234.0,333.0,63.0,7.0,19.720000,7.0,75.0,43.0,291.0,0.0,HE Qualification,successful
2,0.000000,0.0,6.716418,0.000000,59.0,126.0,22.0,6.0,23.416667,4.0,22.0,4.0,66.0,0.0,A Level or Equivalent,at_risk
3,72.333333,3.0,41.977612,-1.305970,297.0,358.0,61.0,7.0,18.426667,11.0,105.0,61.0,549.0,0.0,A Level or Equivalent,successful
4,54.000000,3.0,26.305970,6.529851,152.0,162.0,52.0,7.0,17.000000,32.0,42.0,8.0,401.0,0.0,Lower Than A Level,successful


## Step 2: Check null values
Show the number and percentage of missing values in each column.


In [4]:
null_summary = Selected_features_Q2_copy.isna().sum().to_frame("null_count")
null_summary["null_percentage"] = (null_summary["null_count"] / len(Selected_features_Q2_copy) * 100).round(2)
display(null_summary)


,null_count,null_percentage
avg_score_until_cutoff,0,0.0
submitted_assessments_until_cutoff,0,0.0
arab_active_days_equivalent_until_cutoff,0,0.0
avg_submission_delay_arab_days_until_cutoff,0,0.0
homepage,0,0.0
forumng,0,0.0
unique_sites_until_cutoff,0,0.0
unique_activity_types_until_cutoff,0,0.0
clicks_per_active_day_until_cutoff,0,0.0
resource,0,0.0


## Step 3: Check duplicated rows
Count duplicated rows and preview them if they exist.


In [5]:
duplicated_count = Selected_features_Q2_copy.duplicated().sum()
print(f"Duplicated rows: {duplicated_count}")
display(Selected_features_Q2_copy[Selected_features_Q2_copy.duplicated()].head())


Duplicated rows: 3459


,avg_score_until_cutoff,submitted_assessments_until_cutoff,arab_active_days_equivalent_until_cutoff,avg_submission_delay_arab_days_until_cutoff,homepage,forumng,unique_sites_until_cutoff,unique_activity_types_until_cutoff,clicks_per_active_day_until_cutoff,resource,subpage,url,oucontent,quiz,highest_education,target
196,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,A Level or Equivalent,at_risk
294,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,A Level or Equivalent,at_risk
460,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,A Level or Equivalent,at_risk
506,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,A Level or Equivalent,at_risk
508,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,A Level or Equivalent,at_risk


## Step 4: Check and remove outliers
Use the IQR rule with a 90th percentile upper quantile, then remove rows that contain outliers.


In [6]:
numeric_features = Selected_features_Q2_copy.select_dtypes(include="number").columns.tolist()

q1 = Selected_features_Q2_copy[numeric_features].quantile(0.25)
q3 = Selected_features_Q2_copy[numeric_features].quantile(0.90)
iqr = q3 - q1

outlier_mask = (
    (Selected_features_Q2_copy[numeric_features] < (q1 - 1.5 * iqr)) |
    (Selected_features_Q2_copy[numeric_features] > (q3 + 1.5 * iqr))
)

outlier_summary = outlier_mask.sum().sort_values(ascending=False).to_frame("outlier_count")
outlier_summary["outlier_percentage"] = (outlier_summary["outlier_count"] / len(Selected_features_Q2_copy) * 100).round(2)
display(outlier_summary)

rows_before = len(Selected_features_Q2_copy)
rows_with_outliers = outlier_mask.any(axis=1)
Selected_features_Q2_copy = Selected_features_Q2_copy[~rows_with_outliers].copy()

print(f"Rows before removing outliers: {rows_before}")
print(f"Rows removed: {rows_with_outliers.sum()}")
print(f"Rows after removing outliers: {len(Selected_features_Q2_copy)}")


,outlier_count,outlier_percentage
avg_submission_delay_arab_days_until_cutoff,6197,19.08
forumng,783,2.41
quiz,647,1.99
url,501,1.54
homepage,427,1.31
oucontent,387,1.19
resource,376,1.16
subpage,323,0.99
clicks_per_active_day_until_cutoff,177,0.54
unique_sites_until_cutoff,13,0.04


Rows before removing outliers: 32480
Rows removed: 7767
Rows after removing outliers: 24713


## Step 5: Impute missing values
Use `SimpleImputer`: median for numeric columns and most frequent value for categorical columns.


In [7]:
from sklearn.impute import SimpleImputer

numeric_features = Selected_features_Q2_copy.select_dtypes(include="number").columns.tolist()
categorical_features = Selected_features_Q2_copy.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

if numeric_features:
    num_imputer = SimpleImputer(strategy="median")
    Selected_features_Q2_copy[numeric_features] = num_imputer.fit_transform(Selected_features_Q2_copy[numeric_features])

if categorical_features:
    cat_imputer = SimpleImputer(strategy="most_frequent")
    Selected_features_Q2_copy[categorical_features] = cat_imputer.fit_transform(Selected_features_Q2_copy[categorical_features])

Selected_features_Q2_copy.isna().sum()


avg_score_until_cutoff                         0
submitted_assessments_until_cutoff             0
arab_active_days_equivalent_until_cutoff       0
avg_submission_delay_arab_days_until_cutoff    0
homepage                                       0
forumng                                        0
unique_sites_until_cutoff                      0
unique_activity_types_until_cutoff             0
clicks_per_active_day_until_cutoff             0
resource                                       0
subpage                                        0
url                                            0
oucontent                                      0
quiz                                           0
highest_education                              0
target                                         0
dtype: int64

## Step 6: Encode categorical columns
Use ordinal encoding for ordered education levels, label encoding for the binary target, and one-hot encoding for any remaining nominal categorical columns.


In [8]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder

ordinal_features = ["highest_education"]
education_order = [[
    "No Formal quals",
    "Lower Than A Level",
    "A Level or Equivalent",
    "HE Qualification",
    "Post Graduate Qualification",
]]

if not pd.api.types.is_numeric_dtype(Selected_features_Q2_copy["highest_education"]):
    ordinal_encoder = OrdinalEncoder(categories=education_order)
    Selected_features_Q2_copy[ordinal_features] = ordinal_encoder.fit_transform(
        Selected_features_Q2_copy[ordinal_features]
    ).astype(int)
else:
    print("highest_education is already encoded.")

if not pd.api.types.is_numeric_dtype(Selected_features_Q2_copy["target"]):
    label_encoder = LabelEncoder()
    Selected_features_Q2_copy["target"] = label_encoder.fit_transform(Selected_features_Q2_copy["target"])
    target_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
else:
    target_mapping = "target is already encoded"

nominal_categorical_features = [
    column for column in Selected_features_Q2_copy.select_dtypes(include=["object", "category", "bool"]).columns
    if column not in ordinal_features + ["target"]
]

if nominal_categorical_features:
    try:
        one_hot_encoder = OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)
    except TypeError:
        one_hot_encoder = OneHotEncoder(drop="first", handle_unknown="ignore", sparse=False)

    encoded_values = one_hot_encoder.fit_transform(Selected_features_Q2_copy[nominal_categorical_features])
    encoded_columns = one_hot_encoder.get_feature_names_out(nominal_categorical_features)
    encoded_df = pd.DataFrame(encoded_values, columns=encoded_columns, index=Selected_features_Q2_copy.index)
    Selected_features_Q2_copy = pd.concat(
        [Selected_features_Q2_copy.drop(columns=nominal_categorical_features), encoded_df],
        axis=1,
    )

print("Target mapping:", target_mapping)
display(Selected_features_Q2_copy.head())


Target mapping: {'at_risk': 0, 'successful': 1}


,avg_score_until_cutoff,submitted_assessments_until_cutoff,arab_active_days_equivalent_until_cutoff,avg_submission_delay_arab_days_until_cutoff,homepage,forumng,unique_sites_until_cutoff,unique_activity_types_until_cutoff,clicks_per_active_day_until_cutoff,resource,subpage,url,oucontent,quiz,highest_education,target
0,81.000000,3.0,16.231343,-0.746269,106.0,146.0,43.0,6.0,24.482759,10.0,27.0,2.0,419.0,0.0,3,1
1,69.333333,3.0,27.985075,0.932836,234.0,333.0,63.0,7.0,19.720000,7.0,75.0,43.0,291.0,0.0,3,1
2,0.000000,0.0,6.716418,0.000000,59.0,126.0,22.0,6.0,23.416667,4.0,22.0,4.0,66.0,0.0,2,0
3,72.333333,3.0,41.977612,-1.305970,297.0,358.0,61.0,7.0,18.426667,11.0,105.0,61.0,549.0,0.0,2,1
5,74.000000,3.0,42.537313,1.865672,313.0,540.0,58.0,7.0,17.263158,7.0,53.0,22.0,370.0,0.0,2,1


## Step 7: Normalize features to [0, 1]
Use `MinMaxScaler` to scale numeric feature columns while keeping the target column unchanged.


In [9]:
from sklearn.preprocessing import MinMaxScaler

feature_columns = [column for column in Selected_features_Q2_copy.columns if column != "target"]

scaler = MinMaxScaler()
Selected_features_Q2_copy[feature_columns] = scaler.fit_transform(Selected_features_Q2_copy[feature_columns])

display(Selected_features_Q2_copy.head())


,avg_score_until_cutoff,submitted_assessments_until_cutoff,arab_active_days_equivalent_until_cutoff,avg_submission_delay_arab_days_until_cutoff,homepage,forumng,unique_sites_until_cutoff,unique_activity_types_until_cutoff,clicks_per_active_day_until_cutoff,resource,subpage,url,oucontent,quiz,highest_education,target
0,0.810000,0.375,0.194183,0.432597,0.134008,0.138520,0.212871,0.428571,0.388615,0.074074,0.055785,0.0250,0.266032,0.0,0.75,1
1,0.693333,0.375,0.334799,0.573102,0.295828,0.315939,0.311881,0.500000,0.313016,0.051852,0.154959,0.5375,0.184762,0.0,0.75,1
2,0.000000,0.000,0.080352,0.495044,0.074589,0.119545,0.108911,0.428571,0.371693,0.029630,0.045455,0.0500,0.041905,0.0,0.50,0
3,0.723333,0.375,0.502198,0.385763,0.375474,0.339658,0.301980,0.500000,0.292487,0.081481,0.216942,0.7625,0.348571,0.0,0.50,1
5,0.740000,0.375,0.508894,0.651160,0.395702,0.512334,0.287129,0.500000,0.274018,0.051852,0.109504,0.2750,0.234921,0.0,0.50,1


# Modeling
Start the modeling stage using an 80/20 train-test split and cross-validation.


## Step 8: Split features and target
Separate the input features from the target column.


In [10]:
X = Selected_features_Q2_copy.drop(columns="target")
y = Selected_features_Q2_copy["target"]

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (24713, 15)
y shape: (24713,)


## Step 9: Train-test split
Split the data into 80% training and 20% testing.


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


Train shape: (19770, 15)
Test shape: (4943, 15)


## Step 10: Build pipelines and cross-validation
Create model pipelines for Kernel SVM, Naive Bayes, and XGBoost, then compare them using cross-validation.


In [12]:
%pip install xgboost


Note: you may need to restart the kernel to use updated packages.


In [13]:
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

models = {
    "Kernel SVM": Pipeline([
        ("model", SVC(kernel="rbf", C=1.0, gamma="scale"))
    ]),
    "Naive Bayes": Pipeline([
        ("model", GaussianNB())
    ]),
}

try:
    from xgboost import XGBClassifier

    models["XGBoost"] = Pipeline([
        ("model", XGBClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=3,
            eval_metric="logloss",
            random_state=42,
        ))
    ])
except ImportError:
    print("XGBoost is not installed. Run: pip install xgboost")

cv_results = []

for model_name, model_pipeline in models.items():
    cv_scores = cross_val_score(model_pipeline, X_train, y_train, cv=10, scoring="accuracy")
    cv_results.append({
        "model": model_name,
        "mean_cv_accuracy": cv_scores.mean().round(4),
        "std_cv_accuracy": cv_scores.std().round(4),
    })

cv_results_df = pd.DataFrame(cv_results).sort_values("mean_cv_accuracy", ascending=False)
display(cv_results_df)


,model,mean_cv_accuracy,std_cv_accuracy
2,XGBoost,0.8543,0.0073
0,Kernel SVM,0.8462,0.0067
1,Naive Bayes,0.7791,0.0092


## Step 11: Train and test the models
Fit each pipeline on the training data and evaluate it on the test data.


In [14]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

test_results = []
trained_models = {}

for model_name, model_pipeline in models.items():
    model_pipeline.fit(X_train, y_train)
    y_pred = model_pipeline.predict(X_test)
    trained_models[model_name] = model_pipeline
    test_results.append({
        "model": model_name,
        "test_accuracy": accuracy_score(y_test, y_pred).round(4),
    })

    print(model_name)
    print("Confusion matrix:")
    display(pd.DataFrame(confusion_matrix(y_test, y_pred)))
    print("Classification report:")
    print(classification_report(y_test, y_pred))

test_results_df = pd.DataFrame(test_results).sort_values("test_accuracy", ascending=False)
display(test_results_df)


Kernel SVM
Confusion matrix:


,0,1
0,2482,472
1,264,1725


Classification report:
              precision    recall  f1-score   support

           0       0.90      0.84      0.87      2954
           1       0.79      0.87      0.82      1989

    accuracy                           0.85      4943
   macro avg       0.84      0.85      0.85      4943
weighted avg       0.86      0.85      0.85      4943

Naive Bayes
Confusion matrix:


,0,1
0,2374,580
1,503,1486


Classification report:
              precision    recall  f1-score   support

           0       0.83      0.80      0.81      2954
           1       0.72      0.75      0.73      1989

    accuracy                           0.78      4943
   macro avg       0.77      0.78      0.77      4943
weighted avg       0.78      0.78      0.78      4943

XGBoost
Confusion matrix:


,0,1
0,2497,457
1,252,1737


Classification report:
              precision    recall  f1-score   support

           0       0.91      0.85      0.88      2954
           1       0.79      0.87      0.83      1989

    accuracy                           0.86      4943
   macro avg       0.85      0.86      0.85      4943
weighted avg       0.86      0.86      0.86      4943



,model,test_accuracy
2,XGBoost,0.8566
0,Kernel SVM,0.8511
1,Naive Bayes,0.7809
